# DeepESN and nonlinear readout benchmark

Goal:

1. Test whether reservoir depth improves over shallow ESN.
2. Test whether a stronger readout improves separability.
3. Keep the first experiment narrow: PCA-6 input only.
4. Aggregate across seeds instead of selecting lucky runs.

Suggested interpretation:

- `DeepESN-linear`: deeper reservoir, simple readout.
- `ESN-MLP`: shallow reservoir, nonlinear readout.
- `DeepESN-MLP`: stronger hybrid benchmark.

This is no longer a pure ESN benchmark once MLP readouts are used; it is a stronger classical reservoir benchmark.

In [ ]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

print(Path.cwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qpitome_qrc.data.loaders import load_market_stress_data
from qpitome_qrc.baselines.esn_benchmark import (
    default_esn_feature_sets,
    build_sequence_splits_for_feature_set,
)
from qpitome_qrc.baselines.deep_esn import (
    DeepESNConfig,
    deep_esn_configs,
    fit_deep_esn,
    summarize_deep_esn_run,
)
from qpitome_qrc.baselines.reservoir_readouts import ReadoutConfig
from qpitome_qrc.evaluation.plots import ensure_dir

## 1. Load data and build PCA-6 sequence splits

PCA-6 was the best ESN input representation in the previous benchmark and is also the likely QRC input representation.

In [ ]:
df = load_market_stress_data()
print(df.shape)
df.head()

In [ ]:
feature_sets = default_esn_feature_sets(df, pca_components=6, corr_threshold=0.95)
pca6_feature_set = [fs for fs in feature_sets if fs.name == "compact_pca_6"][0]
pca6_feature_set

In [ ]:
# Start with 20 and 40 day windows, as before.
splits_by_seq_len = {}
feature_names_by_seq_len = {}

for seq_len in (20, 40):
    splits, feature_names = build_sequence_splits_for_feature_set(
        df=df,
        feature_set=pca6_feature_set,
        seq_len=seq_len,
    )
    splits_by_seq_len[seq_len] = splits
    feature_names_by_seq_len[seq_len] = feature_names
    print(seq_len, feature_names)
    for name, (X, y, dates) in splits.items():
        print("  ", name, X.shape, y.shape, dates.min(), dates.max(), "pos_rate", y.mean())

## 2. Experiment A: depth with linear readouts

This tests depth first, without adding a nonlinear output classifier.

Architectures:

- `(300,)`: shallow ESN
- `(300, 300)`: two-layer DeepESN
- `(600,)`: wider shallow ESN
- `(600, 300)`: wide-then-compressed DeepESN

Readouts:

- logistic
- ridge

Use mean validation PR-AUC across seeds for selection.

In [ ]:
depth_configs = deep_esn_configs(
    layer_units=((300,), (300, 300), (600,), (600, 300)),
    spectral_radius=(0.7,),
    leak_rate=(0.5,),
    reservoir_connectivity=(0.1,),
    seeds=(1, 2, 3),
    washout=(0,),
    pooling=("final",),
    state_mode=("last_layer",),
)

linear_readouts = [
    ReadoutConfig(
        kind="logistic",
        scale_states=False,
        class_weight="balanced",
        params={"C": 0.1},
    ),
    ReadoutConfig(
        kind="ridge",
        scale_states=False,
        class_weight="balanced",
        params={"alpha": 1.0},
    ),
]

print("depth configs:", len(depth_configs))
print("readouts:", [r.kind for r in linear_readouts])
print("expected fits:", len(depth_configs) * len(linear_readouts) * len(splits_by_seq_len))

In [ ]:
rows = []
results = {}
run_idx = 0

for seq_len, splits in splits_by_seq_len.items():
    for cfg in depth_configs:
        for readout_cfg in linear_readouts:
            run_idx += 1
            print(f"[{run_idx}] seq_len={seq_len} layers={cfg.layer_units} seed={cfg.seed} readout={readout_cfg.kind}")
            result = fit_deep_esn(
                splits=splits,
                config=cfg,
                readout_config=readout_cfg,
                tune_threshold=True,
            )
            row = summarize_deep_esn_run(result)
            row.update({
                "experiment": "depth_linear",
                "seq_len": seq_len,
                "run_idx": run_idx,
            })
            rows.append(row)
            results[run_idx] = result

depth_summary = pd.DataFrame(rows).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)
depth_summary.head(20)

## 3. Aggregate Experiment A across seeds

In [ ]:
group_cols = [
    "experiment",
    "seq_len",
    "layer_units",
    "spectral_radius",
    "leak_rate",
    "reservoir_connectivity",
    "pooling",
    "state_mode",
    "readout_kind",
    "readout_scale_states",
    "readout_params",
]

depth_agg = (
    depth_summary
    .groupby(group_cols, as_index=False)
    .agg(
        mean_val_pr_auc=("val_pr_auc", "mean"),
        std_val_pr_auc=("val_pr_auc", "std"),
        mean_val_f1=("val_f1_class_1", "mean"),
        mean_val_precision=("val_precision_class_1", "mean"),
        mean_val_recall=("val_recall_class_1", "mean"),
        mean_test_pr_auc=("test_pr_auc", "mean"),
        mean_test_f1=("test_f1_class_1", "mean"),
        n_seeds=("val_pr_auc", "size"),
    )
    .sort_values("mean_val_pr_auc", ascending=False)
    .reset_index(drop=True)
)

depth_agg.head(20)

In [ ]:
plot_df = depth_agg.sort_values("mean_val_pr_auc", ascending=True).tail(20)
labels = (
    plot_df["layer_units"].astype(str)
    + " | T=" + plot_df["seq_len"].astype(str)
    + " | " + plot_df["readout_kind"]
)

plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
plt.barh(labels, plot_df["mean_val_pr_auc"], xerr=plot_df["std_val_pr_auc"].fillna(0))
plt.axvline(0.578, linestyle="--", label="toy tabular reference ~0.578")
plt.axvline(0.600, linestyle=":", label="target 0.60")
plt.xlabel("Mean validation PR-AUC across seeds")
plt.title("Experiment A: depth + linear readout")
plt.legend()
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 4. Experiment B: small reservoir + nonlinear readout

This tests whether the reservoir state contains useful nonlinear structure that a linear readout fails to separate.

Keep this small first. MLP readouts can overfit.

In [ ]:
mlp_reservoir_configs = deep_esn_configs(
    layer_units=((300,), (300, 300)),
    spectral_radius=(0.7,),
    leak_rate=(0.5,),
    reservoir_connectivity=(0.1,),
    seeds=(1, 2, 3),
    washout=(0,),
    pooling=("final",),
    state_mode=("last_layer",),
)

mlp_readouts = [
    ReadoutConfig(
        kind="mlp",
        scale_states=True,
        class_weight=None,
        params={
            "hidden_layer_sizes": (16,),
            "activation": "relu",
            "alpha": 1e-3,
            "learning_rate_init": 1e-3,
            "max_iter": 500,
            "early_stopping": True,
        },
    ),
    ReadoutConfig(
        kind="mlp",
        scale_states=True,
        class_weight=None,
        params={
            "hidden_layer_sizes": (32,),
            "activation": "relu",
            "alpha": 1e-3,
            "learning_rate_init": 1e-3,
            "max_iter": 500,
            "early_stopping": True,
        },
    ),
]

print("reservoir configs:", len(mlp_reservoir_configs))
print("readout configs:", len(mlp_readouts))
print("expected fits:", len(mlp_reservoir_configs) * len(mlp_readouts) * len(splits_by_seq_len))

In [ ]:
mlp_rows = []
mlp_results = {}
mlp_run_idx = 0

for seq_len, splits in splits_by_seq_len.items():
    for cfg in mlp_reservoir_configs:
        for readout_cfg in mlp_readouts:
            mlp_run_idx += 1
            print(f"[{mlp_run_idx}] seq_len={seq_len} layers={cfg.layer_units} seed={cfg.seed} readout={readout_cfg.params}")
            result = fit_deep_esn(
                splits=splits,
                config=cfg,
                readout_config=readout_cfg,
                tune_threshold=True,
            )
            row = summarize_deep_esn_run(result)
            row.update({
                "experiment": "mlp_readout",
                "seq_len": seq_len,
                "run_idx": mlp_run_idx,
            })
            mlp_rows.append(row)
            mlp_results[mlp_run_idx] = result

mlp_summary = pd.DataFrame(mlp_rows).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)
mlp_summary.head(20)

## 5. Aggregate Experiment B

In [ ]:
mlp_agg = (
    mlp_summary
    .groupby(group_cols, as_index=False)
    .agg(
        mean_val_pr_auc=("val_pr_auc", "mean"),
        std_val_pr_auc=("val_pr_auc", "std"),
        mean_val_f1=("val_f1_class_1", "mean"),
        mean_val_precision=("val_precision_class_1", "mean"),
        mean_val_recall=("val_recall_class_1", "mean"),
        mean_test_pr_auc=("test_pr_auc", "mean"),
        mean_test_f1=("test_f1_class_1", "mean"),
        n_seeds=("val_pr_auc", "size"),
    )
    .sort_values("mean_val_pr_auc", ascending=False)
    .reset_index(drop=True)
)

mlp_agg.head(20)

In [ ]:
plot_df = mlp_agg.sort_values("mean_val_pr_auc", ascending=True).tail(20)
labels = (
    plot_df["layer_units"].astype(str)
    + " | T=" + plot_df["seq_len"].astype(str)
    + " | " + plot_df["readout_params"].astype(str)
)

plt.figure(figsize=(11, max(4, 0.35 * len(plot_df))))
plt.barh(labels, plot_df["mean_val_pr_auc"], xerr=plot_df["std_val_pr_auc"].fillna(0))
plt.axvline(0.578, linestyle="--", label="toy tabular reference ~0.578")
plt.axvline(0.600, linestyle=":", label="target 0.60")
plt.xlabel("Mean validation PR-AUC across seeds")
plt.title("Experiment B: nonlinear MLP readout")
plt.legend()
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 6. Combined comparison

In [ ]:
combined_agg = pd.concat([depth_agg, mlp_agg], ignore_index=True)
combined_agg = combined_agg.sort_values("mean_val_pr_auc", ascending=False).reset_index(drop=True)

display_cols = [
    "experiment",
    "seq_len",
    "layer_units",
    "readout_kind",
    "readout_params",
    "mean_val_pr_auc",
    "std_val_pr_auc",
    "mean_val_f1",
    "mean_val_precision",
    "mean_val_recall",
    "mean_test_pr_auc",
    "mean_test_f1",
    "n_seeds",
]
combined_agg[display_cols].head(30)

In [ ]:
plot_df = combined_agg.sort_values("mean_val_pr_auc", ascending=True).tail(25)
labels = (
    plot_df["experiment"]
    + " | " + plot_df["layer_units"].astype(str)
    + " | T=" + plot_df["seq_len"].astype(str)
    + " | " + plot_df["readout_kind"]
)

plt.figure(figsize=(11, max(5, 0.35 * len(plot_df))))
plt.barh(labels, plot_df["mean_val_pr_auc"], xerr=plot_df["std_val_pr_auc"].fillna(0))
plt.axvline(0.578, linestyle="--", label="toy tabular reference ~0.578")
plt.axvline(0.600, linestyle=":", label="target 0.60")
plt.xlabel("Mean validation PR-AUC across seeds")
plt.title("DeepESN/readout benchmark summary")
plt.legend()
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Save outputs

In [ ]:
OUT_DIR = Path("reports/deep_esn_benchmark/tables")
ensure_dir(OUT_DIR)

depth_summary.to_csv(OUT_DIR / "depth_linear_runs.csv", index=False)
depth_agg.to_csv(OUT_DIR / "depth_linear_seed_aggregate.csv", index=False)
mlp_summary.to_csv(OUT_DIR / "mlp_readout_runs.csv", index=False)
mlp_agg.to_csv(OUT_DIR / "mlp_readout_seed_aggregate.csv", index=False)
combined_agg.to_csv(OUT_DIR / "combined_seed_aggregate.csv", index=False)

print(OUT_DIR)

## 8. Interpretation scratchpad

Questions:

1. Does two-layer DeepESN beat shallow ESN?
2. Does ridge readout beat logistic?
3. Does MLP readout improve validation PR-AUC or just overfit?
4. Does precision improve, or is recall still carrying the result?
5. Does any configuration beat the toy tabular reference (~0.578 val PR-AUC)?
6. Does anything reach the target ~0.60 val PR-AUC?

If nothing beats the toy tabular reference, the next honest move is HMM/regime modeling or target redesign, not more reservoir tuning.